# More coverage: 8 centers and up to 2 series per plane

Same 5-fold CoAtNet model and training as the 6-center, 1-series baseline. Select one series per available plane and a second when present (prefer different fluid/fat metadata), then form 8 three-slice inputs per series. At most 48 inputs per study, using at most 144 distinct DICOM slices. Per-label top-2 pooling still selects among all inputs for a plane; the second series can contribute but is not guaranteed to. More coverage costs up to 2.67 times as many forward passes and may worsen performance.

After every epoch, evaluate the fold's held-out studies and retain the checkpoint with lowest validation BCE. Also log macro AUC on labels with both classes in that fold. **Because the same held-out fold selects the epoch, the resulting OOF AUC is optimistically biased; use the independent leaderboard to judge the experiment.** No independent runtime test has been run here.

Train 5 gold folds, save checkpoints and OOF, then produce `submission.csv`. Keep the original 0.637 score as the comparison.


In [1]:
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
import time, random, gc
import numpy as np
import pandas as pd
import torch, cv2, pydicom, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark=True
LABEL_COLS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
IMAGE_SIZE=224; CENTERS_PER_PLANE=8; IMAGE_MICROBATCH=3
NUM_DICOM_THREADS=8; NUM_WORKERS=2; N_FOLDS=5; EPOCHS=5
LR_BACKBONE=1e-5; LR_HEAD=2e-4
DATA_CANDIDATES=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
DATA_ROOT=next((p for p in DATA_CANDIDATES if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('Attach RSNA Knee Abnormality Detection competition data.')
TRAIN_SERIES_ROOT=DATA_ROOT/'train_series'; TEST_SERIES_ROOT=DATA_ROOT/'test_series'
OUT=Path('/kaggle/working'); CACHE=OUT/'coatnet_2p5d_morecoverage_cache'; CACHE.mkdir(exist_ok=True)
COATNET_WEIGHTS= Path('/kaggle/input/models/amritgopinath/coatnet-model/pytorch/default/1/model.safetensors')
print('device',DEVICE,'timm',timm.__version__)


device cuda timm 1.0.26


In [2]:
def find_coatnet():
    for top in Path('/kaggle/input').iterdir():
        if top.is_dir() and 'coatnet' in top.name.lower():
            for name in ('model.safetensors','pytorch_model.bin','coatnet_0_rw_224.pth'):
                matches=list(top.rglob(name))
                if matches: return matches[0]
    return None
COATNET_WEIGHTS=Path(COATNET_WEIGHTS) if COATNET_WEIGHTS else find_coatnet()
if COATNET_WEIGHTS is None: raise FileNotFoundError('Attach official timm coatnet_0_rw_224.sw_in1k weights, then set COATNET_WEIGHTS.')
print('ImageNet weights',COATNET_WEIGHTS)


ImageNet weights /kaggle/input/models/amritgopinath/coatnet-model/pytorch/default/1/model.safetensors


In [3]:
train=pd.read_csv(DATA_ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(DATA_ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test_series=pd.read_csv(DATA_ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample_submission=pd.read_csv(DATA_ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert list(sample_submission.columns)==['StudyInstanceUID']+LABEL_COLS
gold=train[train[LABEL_COLS].notna().all(axis=1)].copy().reset_index(drop=True)
gold[LABEL_COLS]=gold[LABEL_COLS].apply(pd.to_numeric)
assert len(gold)==58, f'Expected 58 gold studies; got {len(gold)}'
assert gold.StudyInstanceUID.is_unique
print('gold studies',len(gold),'test studies',len(sample_submission))


gold studies 58 test studies 3


In [4]:
def plane_of(row):
    p=str(row.get('Anatomical_Plane','')).strip().upper()
    if p.startswith('SAG'): return 'SAGITTAL'
    if p.startswith('COR'): return 'CORONAL'
    if p.startswith('AX'): return 'AXIAL'
    return 'UNKNOWN'

def select_series(df, ids):
    df=df.copy()
    df['StudyInstanceUID']=df.StudyInstanceUID.astype(str)
    df['SeriesInstanceUID']=df.SeriesInstanceUID.astype(str)
    df=df[df.StudyInstanceUID.isin(set(map(str,ids)))].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    df['plane_norm']=df.apply(plane_of,axis=1)
    for col in ('Fluid_Sensitive','Fat_Suppression'):
        if col not in df: df[col]='UNKNOWN'
        df[col]=df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    rows=[]
    for sid,group in df.groupby('StudyInstanceUID',sort=True):
        group=group.sort_values('SeriesInstanceUID')
        picks=[]
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            if len(candidates): picks.append(candidates.iloc[0])
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            used=[p for p in picks if p.plane_norm==plane]
            if len(candidates)<2 or not used: continue
            first=used[0]
            rest=candidates[candidates.SeriesInstanceUID.ne(first.SeriesInstanceUID)]
            diverse=rest[(rest.Fluid_Sensitive.ne(first.Fluid_Sensitive)) | (rest.Fat_Suppression.ne(first.Fat_Suppression))]
            if len(diverse): picks.append(diverse.iloc[0])
            elif len(rest): picks.append(rest.iloc[0])
        if not picks and len(group): picks.append(group.iloc[0])
        rows.extend([{'StudyInstanceUID':sid,'SeriesInstanceUID':str(r.SeriesInstanceUID),'plane_norm':r.plane_norm} for r in picks])
    return pd.DataFrame(rows,columns=['StudyInstanceUID','SeriesInstanceUID','plane_norm'])

train_picks=select_series(train_series,gold.StudyInstanceUID)
test_picks=select_series(test_series,sample_submission.StudyInstanceUID)
train_picks.to_csv(OUT/'coatnet_morecoverage_gold_series_selection.csv',index=False)
assert set(gold.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Gold study has no metadata series'
print('gold series',len(train_picks),'test series',len(test_picks))


gold series 291 test series 14


In [5]:
def _read_one(path):
    try:
        d=pydicom.dcmread(str(path),force=True)
        x=np.squeeze(d.pixel_array)
        if x.ndim!=2: return None
        x=x.astype(np.float32)*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
        if str(getattr(d,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        o=np.asarray(getattr(d,'ImageOrientationPatient',[]),float)
        pos=np.asarray(getattr(d,'ImagePositionPatient',[]),float)
        g=np.nan
        if len(o)==6 and len(pos)==3:
            n=np.cross(o[:3],o[3:]); norm=np.linalg.norm(n)
            if norm>0: g=float(pos @ (n/norm))
        try: inst=float(d.InstanceNumber)
        except Exception: inst=np.nan
        return (path.name,g,inst,x)
    except Exception: return None

def resize_pad(x):
    h,w=x.shape; scale=IMAGE_SIZE/max(h,w)
    nh,nw=max(1,round(h*scale)),max(1,round(w*scale))
    z=cv2.resize(x,(nw,nh),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_LINEAR)
    out=np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.uint8)
    top=(IMAGE_SIZE-nh)//2; left=(IMAGE_SIZE-nw)//2
    out[top:top+nh,left:left+nw]=np.rint(z*255).astype(np.uint8)
    return out

def load_plane(study,series,root):
    directory=root/str(study)/str(series)
    paths=sorted(p for p in directory.iterdir() if p.is_file() and not p.name.startswith('.')) if directory.is_dir() else []
    if not paths: return None
    with ThreadPoolExecutor(max_workers=NUM_DICOM_THREADS) as pool: rows=[r for r in pool.map(_read_one,paths) if r is not None]
    if not rows: return None
    key=1 if all(np.isfinite(r[1]) for r in rows) else 2 if all(np.isfinite(r[2]) for r in rows) else 0
    rows.sort(key=lambda r:(r[key],r[0]) if key else (r[0],))
    samples=np.concatenate([r[3][::8,::8].ravel() for r in rows])
    samples=samples[np.isfinite(samples)]
    if not len(samples): return None
    lo,hi=np.percentile(samples,[1,99])
    n=len(rows); centers=np.rint(np.linspace(.05*(n-1),.95*(n-1),CENTERS_PER_PLANE)).astype(int)
    needed=sorted(set(np.clip(np.r_[centers-1,centers,centers+1],0,n-1)))
    maps={i:resize_pad(np.clip(np.nan_to_num(rows[i][3],nan=lo,posinf=hi,neginf=lo),lo,hi).astype(np.float32).clip(lo,hi).astype(np.float32)/(hi-lo) if hi>lo else np.zeros_like(rows[i][3],np.float32)) for i in needed}
    return np.stack([np.stack([maps[max(0,i-1)],maps[i],maps[min(n-1,i+1)]],0) for i in centers],0)

def cache_path(study,root):
    return CACHE/('train_' if root==TRAIN_SERIES_ROOT else 'test_')/f'{study}.npz'

def cache_study(study,picks,root):
    path=cache_path(study,root)
    if path.exists(): return path
    blocks=[]; planes=[]
    group=picks[picks.StudyInstanceUID.eq(str(study))]
    for row in group.itertuples(index=False):
        try: block=load_plane(study,row.SeriesInstanceUID,root)
        except Exception: block=None
        if block is None: continue
        blocks.append(block); planes.extend([('SAGITTAL','CORONAL','AXIAL').index(row.plane_norm) if row.plane_norm in ('SAGITTAL','CORONAL','AXIAL') else 3]*len(block))
    if not blocks: return None
    path.parent.mkdir(parents=True,exist_ok=True)
    np.savez(path,images=np.concatenate(blocks,0),planes=np.asarray(planes,np.int8))
    return path

def warm_cache(ids,picks,root):
    t=time.time(); failures=[]
    for i,sid in enumerate(map(str,ids),1):
        if cache_study(sid,picks,root) is None: failures.append(sid)
        if i%100==0 or i==len(ids): print(f'cached {i}/{len(ids)} in {(time.time()-t)/60:.1f} min; failed={len(failures)}')
    return failures


gold_failures=warm_cache(gold.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
assert not gold_failures, f'Gold studies failed DICOM decode: {gold_failures}'


cached 58/58 in 1.2 min; failed=0


In [6]:
from safetensors.torch import load_file as load_safetensors

def make_backbone(load_pretrained=True):
    net=timm.create_model('coatnet_0_rw_224.sw_in1k',pretrained=False)
    if load_pretrained and COATNET_WEIGHTS.suffix=='.safetensors': state=load_safetensors(str(COATNET_WEIGHTS),device='cpu')
    elif load_pretrained:
        state=torch.load(COATNET_WEIGHTS,map_location='cpu',weights_only=True)
        if isinstance(state,dict): state=state.get('state_dict',state.get('model',state))
        state={k.removeprefix('module.'):v for k,v in state.items()}
    if load_pretrained: net.load_state_dict(state,strict=True)
    net.reset_classifier(0)
    for p in net.parameters(): p.requires_grad_(False)
    if not hasattr(net,'stages'): raise RuntimeError('Unexpected timm CoAtNet layout; inspect net and choose its final stage.')
    for p in net.stages[-1].parameters(): p.requires_grad_(True)
    return net

class StudyCoAtNet(nn.Module):
    def __init__(self,load_pretrained=True):
        super().__init__()
        self.backbone=make_backbone(load_pretrained)
        dim=self.backbone.num_features
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(.15),nn.Linear(dim,len(LABEL_COLS)))
        self.plane_weight=nn.Parameter(torch.zeros(4,len(LABEL_COLS)))
        self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))

    def forward(self,images,planes):
        slice_logits=[]
        for chunk in images.split(IMAGE_MICROBATCH):
            x=(chunk.to(dtype=self.mean.dtype)/255-self.mean)/self.std
            ctx=torch.autocast('cuda',dtype=torch.float16) if self.training and images.is_cuda else nullcontext()
            with ctx: slice_logits.append(self.head(self.backbone(x)))
        scores=torch.cat(slice_logits,0)
        pooled=[]; available=[]
        for plane in range(4):
            part=scores[planes==plane]
            available.append(len(part)>0)
            pooled.append(part.topk(min(2,len(part)),dim=0).values.mean(0) if len(part) else torch.zeros_like(scores[0]))
        mask=torch.tensor(available,device=images.device)
        weights=self.plane_weight.masked_fill(~mask[:,None],-1e4).softmax(0)
        return (weights*torch.stack(pooled)).sum(0)


In [7]:
class BagDS(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with np.load(cache_path(str(row.StudyInstanceUID),TRAIN_SERIES_ROOT)) as d:
            x=d['images'].copy(); plane=d['planes'].astype(np.int64)
        return torch.from_numpy(x),torch.from_numpy(plane),torch.tensor(row[LABEL_COLS].to_numpy(np.float32))

def train_fold(model,frame,validation,fold,checkpoint):
    loader=DataLoader(BagDS(frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                      collate_fn=lambda b:b[0])
    optimizer=torch.optim.AdamW([{'params':model.backbone.stages[-1].parameters(),'lr':LR_BACKBONE},
                                 {'params':list(model.head.parameters())+[model.plane_weight],'lr':LR_HEAD}],weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
    best_bce=np.inf; best_epoch=None; history=[]
    for epoch in range(1,EPOCHS+1):
        model.train(); model.backbone.eval(); model.backbone.stages[-1].train()
        losses=[]; start=time.time()
        for x,plane,y in loader:
            x=x.to(DEVICE,non_blocking=True); plane=plane.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,plane)
            loss=nn.functional.binary_cross_entropy_with_logits(logits.float(),y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
        pred,failed=predict_ids(model,validation.StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        if failed: raise RuntimeError(f'Fold {fold} validation decode failures: {failed}')
        aligned=validation[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        y=validation[LABEL_COLS].to_numpy(np.float32)
        probs=aligned[LABEL_COLS].to_numpy(np.float32).clip(1e-6,1-1e-6)
        val_bce=nn.functional.binary_cross_entropy(torch.from_numpy(probs),torch.from_numpy(y)).item()
        aucs=[roc_auc_score(y[:,j],probs[:,j]) for j in range(len(LABEL_COLS)) if np.unique(y[:,j]).size==2]
        val_auc=float(np.mean(aucs)) if aucs else float('nan')
        improved=val_bce<best_bce-1e-6
        if improved:
            best_bce=val_bce; best_epoch=epoch
            torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},checkpoint)
        history.append({'fold':fold,'epoch':epoch,'train_bce':float(np.mean(losses)),
                        'val_bce':val_bce,'val_macro_auc':val_auc,'n_auc_labels':len(aucs),
                        'best_so_far':best_epoch,'minutes':(time.time()-start)/60})
        print(f'fold {fold} epoch {epoch}/{EPOCHS}: train BCE {np.mean(losses):.4f}, '
              f'val BCE {val_bce:.4f}, val AUC {val_auc:.4f} ({len(aucs)} labels), '
              f'best epoch {best_epoch}, time {(time.time()-start)/60:.1f}m')
    model.load_state_dict(torch.load(checkpoint,map_location='cpu',weights_only=True),strict=True)
    return best_epoch,best_bce,history

@torch.inference_mode()
def predict_ids(model,ids,picks,root):
    model.eval(); rows=[]; failed=[]
    for sid in map(str,ids):
        path=cache_path(sid,root)
        if not path.exists(): cache_study(sid,picks,root)
        if not path.exists(): failed.append(sid); continue
        with np.load(path) as d:
            x=torch.from_numpy(d['images'].copy()).to(DEVICE); plane=torch.from_numpy(d['planes'].astype(np.int64)).to(DEVICE)
        score=torch.sigmoid(model(x,plane).float()).cpu().numpy()
        rows.append([sid,*score.tolist()])
    return pd.DataFrame(rows,columns=['StudyInstanceUID']+LABEL_COLS),failed

folds=KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
oof=np.full((len(gold),len(LABEL_COLS)),np.nan,np.float32)
fold_scores=[]; epoch_history=[]
for fold,(tr,va) in enumerate(folds.split(gold),1):
    print('\nFOLD',fold,'train',len(tr),'validation',len(va))
    model=StudyCoAtNet(load_pretrained=True).to(DEVICE)
    checkpoint=OUT/f'coatnet_morecoverage_gold_fold{fold}.pt'
    best_epoch,best_bce,history=train_fold(model,gold.iloc[tr],gold.iloc[va],fold,checkpoint)
    epoch_history.extend(history)
    pred,failed=predict_ids(model,gold.iloc[va].StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
    assert not failed
    aligned=gold.iloc[va][['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
    oof[va]=aligned[LABEL_COLS].to_numpy(np.float32)
    fold_scores.append({'fold':fold,'n_train':len(tr),'n_validation':len(va),'checkpoint':str(checkpoint),'best_epoch':best_epoch,'best_val_bce':best_bce})
    print('saved',checkpoint)
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
assert np.isfinite(oof).all()
oof_df=gold[['StudyInstanceUID']+LABEL_COLS].copy()
for j,label in enumerate(LABEL_COLS): oof_df['pred__'+label]=oof[:,j]
oof_df.to_csv(OUT/'coatnet_morecoverage_oof_58.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/'coatnet_morecoverage_fold_manifest.csv',index=False)
pd.DataFrame(epoch_history).to_csv(OUT/'coatnet_morecoverage_epoch_history.csv',index=False)
metrics=[]
for j,label in enumerate(LABEL_COLS):
    y=gold[label].to_numpy(float)
    metrics.append({'label':label,'positives':int(y.sum()),'auc':roc_auc_score(y,oof[:,j]) if np.unique(y).size==2 else np.nan})
metrics=pd.DataFrame(metrics); metrics.to_csv(OUT/'coatnet_morecoverage_oof_auc.csv',index=False)
display(metrics); print('5-fold OOF macro AUC:',metrics.auc.mean())



FOLD 1 train 46 validation 12


/tmp/ipykernel_23/548398457.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 1 epoch 1/5: train BCE 0.6804, val BCE 0.5757, val AUC 0.5092 (10 labels), best epoch 1, time 0.4m
fold 1 epoch 2/5: train BCE 0.6130, val BCE 0.5651, val AUC 0.5763 (10 labels), best epoch 2, time 0.2m
fold 1 epoch 3/5: train BCE 0.5899, val BCE 0.5720, val AUC 0.5835 (10 labels), best epoch 2, time 0.2m
fold 1 epoch 4/5: train BCE 0.5633, val BCE 0.5798, val AUC 0.5961 (10 labels), best epoch 2, time 0.2m
fold 1 epoch 5/5: train BCE 0.5346, val BCE 0.5842, val AUC 0.5960 (10 labels), best epoch 2, time 0.2m
saved /kaggle/working/coatnet_morecoverage_gold_fold1.pt

FOLD 2 train 46 validation 12


/tmp/ipykernel_23/548398457.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 2 epoch 1/5: train BCE 0.6837, val BCE 0.6614, val AUC 0.4559 (11 labels), best epoch 1, time 0.2m
fold 2 epoch 2/5: train BCE 0.6054, val BCE 0.6381, val AUC 0.4711 (11 labels), best epoch 2, time 0.2m
fold 2 epoch 3/5: train BCE 0.5868, val BCE 0.6471, val AUC 0.4863 (11 labels), best epoch 2, time 0.2m
fold 2 epoch 4/5: train BCE 0.5675, val BCE 0.6333, val AUC 0.5050 (11 labels), best epoch 4, time 0.2m
fold 2 epoch 5/5: train BCE 0.5390, val BCE 0.6467, val AUC 0.5168 (11 labels), best epoch 4, time 0.2m
saved /kaggle/working/coatnet_morecoverage_gold_fold2.pt

FOLD 3 train 46 validation 12


...

fold 4 epoch 1/5: train BCE 0.6645, val BCE 0.6465, val AUC 0.5382 (12 labels), best epoch 1, time 0.2m
fold 4 epoch 2/5: train BCE 0.6121, val BCE 0.6306, val AUC 0.5698 (12 labels), best epoch 2, time 0.2m
fold 4 epoch 3/5: train BCE 0.5865, val BCE 0.6209, val AUC 0.5756 (12 labels), best epoch 3, time 0.3m
fold 4 epoch 4/5: train BCE 0.5665, val BCE 0.6151, val AUC 0.5973 (12 labels), best epoch 4, time 0.2m
fold 4 epoch 5/5: train BCE 0.5422, val BCE 0.6138, val AUC 0.6099 (12 labels), best epoch 5, time 0.2m
saved /kaggle/working/coatnet_morecoverage_gold_fold4.pt

FOLD 5 train 47 validation 11


/tmp/ipykernel_23/548398457.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 5 epoch 1/5: train BCE 0.6494, val BCE 0.7166, val AUC 0.5927 (12 labels), best epoch 1, time 0.2m
fold 5 epoch 2/5: train BCE 0.6096, val BCE 0.6763, val AUC 0.6188 (12 labels), best epoch 2, time 0.2m
fold 5 epoch 3/5: train BCE 0.5784, val BCE 0.6740, val AUC 0.6254 (12 labels), best epoch 3, time 0.2m
fold 5 epoch 4/5: train BCE 0.5562, val BCE 0.6674, val AUC 0.6763 (12 labels), best epoch 4, time 0.2m
fold 5 epoch 5/5: train BCE 0.5375, val BCE 0.6450, val AUC 0.6703 (12 labels), best epoch 5, time 0.2m
saved /kaggle/working/coatnet_morecoverage_gold_fold5.pt


,label,positives,auc
0,ACL,24,0.424020
1,MCL,9,0.414966
2,Medial Meniscus,26,0.514423
3,Lateral Meniscus,23,0.575155
4,Medial OA,15,0.728682
5,Lateral OA,11,0.558994
6,PF OA,21,0.670528
7,Effusion,35,0.650932
8,Synovitis,27,0.507766
9,Baker's,12,0.621377


5-fold OOF macro AUC: 0.5477658925870412


In [8]:
# The test DICOMs are decoded and cached once; the five fold checkpoints are used sequentially.
test_ids=sample_submission.StudyInstanceUID.astype(str).tolist()
test_failures=warm_cache(test_ids,test_picks,TEST_SERIES_ROOT)
print('test decode failures',len(test_failures))
ensemble=np.zeros((len(test_ids),len(LABEL_COLS)),np.float64)
for fold in range(1,N_FOLDS+1):
    model=StudyCoAtNet(load_pretrained=False)
    state=torch.load(OUT/f'coatnet_morecoverage_gold_fold{fold}.pt',map_location='cpu',weights_only=True)
    model.load_state_dict(state,strict=True)
    model=model.to(DEVICE)
    if DEVICE.type=='cuda': model=model.half()  # half weights and half inputs, no autocast at inference
    pred,failed=predict_ids(model,test_ids,test_picks,TEST_SERIES_ROOT)
    aligned=sample_submission[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',how='left',validate='one_to_one')
    fallback=gold[LABEL_COLS].mean().to_numpy(float)
    for j,label in enumerate(LABEL_COLS): aligned[label]=aligned[label].fillna(float(fallback[j]))
    ensemble+=aligned[LABEL_COLS].to_numpy(float)/N_FOLDS
    print('fold',fold,'predicted',len(pred),'missing',len(failed))
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
submission=sample_submission[['StudyInstanceUID']].copy()
submission[LABEL_COLS]=ensemble.astype(np.float32)
assert list(submission.columns)==list(sample_submission.columns)
assert submission.StudyInstanceUID.is_unique and len(submission)==len(sample_submission)
assert np.isfinite(ensemble).all() and ensemble.min()>=0 and ensemble.max()<=1
submission.to_csv(OUT/'submission.csv',index=False)
print('wrote',OUT/'submission.csv','shape',submission.shape)
display(submission.head())


cached 3/3 in 0.1 min; failed=0
test decode failures 0
fold 1 predicted 3 missing 0
fold 2 predicted 3 missing 0
fold 3 predicted 3 missing 0
fold 4 predicted 3 missing 0
fold 5 predicted 3 missing 0
wrote /kaggle/working/submission.csv shape (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.420819,0.158416,0.515969,0.367232,0.303082,0.237545,0.421279,0.544594,0.464895,0.259663,0.291093,0.246813
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.386312,0.155977,0.365810,0.312247,0.206488,0.175619,0.318925,0.543376,0.365931,0.177114,0.303509,0.239662
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.375101,0.183489,0.417462,0.357771,0.244799,0.191494,0.301304,0.536118,0.383341,0.174491,0.287129,0.238297
